# UMAP clustering notebook

## Purpose

This notebook implements an independent clustering approach to validate the distance-space results from the weighted notebook. Rather than clustering on a precomputed pairwise distance matrix, it first compresses the data into a low-dimensional latent space using UMAP, then applies k-medoids in that compressed representation.

The goal is not to replace the distance-space result but to stress-test it. If two fundamentally different methods assign the same clients to the same groups, that is strong evidence the segments are real and not an artefact of the distance function chosen.

## Why UMAP before clustering?

Distance-space clustering operates in the full 17-feature space. This is theoretically correct but high-dimensional spaces are sparse — distances between points tend to concentrate, and the ratio between the nearest and farthest neighbours shrinks as dimensionality grows. This is the curse of dimensionality, and it can make cluster boundaries less sharp even when genuine structure exists.

UMAP learns a low-dimensional manifold that preserves local neighbourhood structure. Clustering in 8–15 dimensional latent space operates where neighbourhood geometry is more faithful and distance concentration is less severe. The trade-off is that UMAP is a non-linear, stochastic compression — it discards information and different random seeds produce slightly different embeddings.

## Is UMAP clustering better?

Not necessarily, and the silhouette scores cannot be directly compared. The distance-space silhouette is computed on the mixed distance matrix; the UMAP silhouette is computed on Euclidean distances in latent space. A higher UMAP silhouette means clusters are more separated in the compressed representation — which UMAP was explicitly designed to produce — not that the segmentation is more correct. Comparing the two values is like comparing a score in metres to a score in kilograms.

The meaningful comparison is the ARI in the comparison notebook.

| ARI | Interpretation |
|---|---|
| ≥ 0.85 | Near-perfect agreement |
| 0.60–0.85 | Strong agreement, some boundary disagreement |
| 0.40–0.60 | Partial agreement, methods diverge on several clients |
| 0.00–0.40 | Weak agreement, investigate before reporting |
| < 0.00 | Worse than random |

## Stability

UMAP is stochastic. The pipeline is re-run five times with different seeds and the ARI between each pair of runs is measured. A mean ARI ≥ 0.85 means the cluster structure does not depend on the random initialisation and can be reported with confidence. A configuration with slightly lower silhouette but ARI ≥ 0.85 is always preferable to one with higher silhouette but ARI < 0.85 — an unstable result is not a finding.

## What to report

- **ARI ≥ 0.70:** Report weighted as primary, cite UMAP as validation. The segments are robust.
- **ARI 0.40–0.70:** Report both side by side. Identify the stable core of each cluster and flag boundary clients for closer inspection.
- **ARI < 0.40:** Do not report either as final. Return to the distance metric analysis.

In [2]:
import numpy as np
import pandas as pd
import pickle
import logging
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.ensemble import IsolationForest
from scipy.spatial.distance import cdist
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score,
    calinski_harabasz_score, adjusted_rand_score,
)
import kmedoids
import gower as gower_lib
import umap
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import sys
sys.path.append(str(Path.cwd()))
from distance import (
    mixed_distance_matrix,
    build_gower_cat_mask,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s — %(levelname)s — %(message)s",
)
logger = logging.getLogger(__name__)

In [3]:
df_raw = pd.read_excel(
    r"C:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\Data\Dataset1_BankClients.xlsx",
    nrows=500  # FOR DEBUGGING, ONLY FIRST 50 INSTANCES
)
print(f"Raw data: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
df_raw.head()


Raw data: 500 rows × 18 columns


,ID,Age,Gender,Job,Area,CitySize,FamilySize,Income,Wealth,Debt,FinEdu,ESG,Digital,BankFriend,LifeStyle,Luxury,Saving,Investments
0,1,24,1,1,2,2,4,0.668046,0.702786,0.262070,0.741853,0.483684,0.698625,0.618259,0.607877,0.897369,0.283222,1
1,2,47,1,2,2,3,1,0.858453,0.915043,0.730430,0.859423,0.537167,0.959025,0.785936,0.862271,0.913729,0.821590,3
2,3,38,0,2,1,2,2,0.926818,0.898316,0.441272,0.485953,0.649434,0.750265,0.699725,0.755404,0.765199,0.503790,3
3,4,67,0,2,1,2,3,0.538797,0.423180,0.600401,0.493144,0.533829,0.590165,0.675353,0.334432,0.517209,0.691240,2
4,5,33,0,2,1,3,1,0.806659,0.731404,0.831449,0.856286,0.784940,0.710026,0.758793,0.908878,0.611610,0.615916,2


In [4]:
# Preprocessing + input arrays
categorical_cols = ["Gender", "Job", "Area"]
numerical_cols   = [c for c in df_raw.columns if c not in categorical_cols + ["ID"]]

logger.info("Numerical columns (%d): %s", len(numerical_cols), numerical_cols)
logger.info("Categorical columns (%d): %s", len(categorical_cols), categorical_cols)

df = df_raw.drop(columns=["ID"])

for col in numerical_cols:
    n_missing = df[col].isna().sum()
    if n_missing:
        logger.info("Imputing %d missing values in '%s' with median.", n_missing, col)
    df[col] = df[col].fillna(df[col].median())

for col in categorical_cols:
    n_missing = df[col].isna().sum()
    if n_missing:
        logger.info("Imputing %d missing values in '%s' with mode.", n_missing, col)
    df[col] = df[col].fillna(df[col].mode()[0])

assert df.isna().sum().sum() == 0, "Missing values remain after imputation!"
logger.info("Missing values after imputation: 0")

mask_minors = (df["Age"] < 18) & (df["Job"].isin([2, 3, 4]))
n_removed   = mask_minors.sum()
df = df[~mask_minors].reset_index(drop=True)
logger.info("Minor filter: removed %d rows → %d remaining.", n_removed, len(df))

iso    = IsolationForest(contamination=0.01, random_state=42)
labels = iso.fit_predict(df[numerical_cols])
n_out  = (labels == -1).sum()
df     = df[labels == 1].reset_index(drop=True)
logger.info("Isolation Forest: removed %d outliers → %d remaining.", n_out, len(df))

scaler = MinMaxScaler()
X_num  = scaler.fit_transform(df[numerical_cols]).astype(float)
logger.info("X_num scaled to [0,1]: min=%.4f  max=%.4f", X_num.min(), X_num.max())

le_encoders = {}
X_cat_int   = np.zeros((len(df), len(categorical_cols)), dtype=np.int32)
for j, col in enumerate(categorical_cols):
    le = LabelEncoder()
    X_cat_int[:, j] = le.fit_transform(df[col])
    le_encoders[col] = le
    logger.info("LabelEncoder '%s': %d classes → %s",
                col, len(le.classes_), list(le.classes_))

X_cat_ohe = pd.get_dummies(df[categorical_cols].astype(str)).values.astype(np.int32)
logger.info("One-hot encoded shape: %s  unique values: %s",
            X_cat_ohe.shape, np.unique(X_cat_ohe).tolist())

n_num = X_num.shape[1]
alpha_neutral_hamming  = n_num / (n_num + X_cat_int.shape[1])
alpha_neutral_tanimoto = n_num / (n_num + X_cat_ohe.shape[1])

print(f"X_num shape     : {X_num.shape}")
print(f"X_cat_int shape : {X_cat_int.shape}")
print(f"X_cat_ohe shape : {X_cat_ohe.shape}")

2026-03-19 21:21:10,347 — INFO — Numerical columns (14): ['Age', 'CitySize', 'FamilySize', 'Income', 'Wealth', 'Debt', 'FinEdu', 'ESG', 'Digital', 'BankFriend', 'LifeStyle', 'Luxury', 'Saving', 'Investments']
2026-03-19 21:21:10,348 — INFO — Categorical columns (3): ['Gender', 'Job', 'Area']
2026-03-19 21:21:10,372 — INFO — Missing values after imputation: 0
2026-03-19 21:21:10,375 — INFO — Minor filter: removed 0 rows → 500 remaining.
2026-03-19 21:21:10,634 — INFO — Isolation Forest: removed 5 outliers → 495 remaining.
2026-03-19 21:21:10,640 — INFO — X_num scaled to [0,1]: min=0.0000  max=1.0000
2026-03-19 21:21:10,642 — INFO — LabelEncoder 'Gender': 2 classes → [np.int64(0), np.int64(1)]
2026-03-19 21:21:10,644 — INFO — LabelEncoder 'Job': 5 classes → [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
2026-03-19 21:21:10,644 — INFO — LabelEncoder 'Area': 3 classes → [np.int64(1), np.int64(2), np.int64(3)]
2026-03-19 21:21:10,662 — INFO — One-hot encoded shape: (495, 

X_num shape     : (495, 14)
X_cat_int shape : (495, 3)
X_cat_ohe shape : (495, 10)


In [5]:
# Load best config from weighted notebook
RESULTS_DIR = Path("results")

with open(RESULTS_DIR / "weighted_results.pkl", "rb") as f:
    weighted = pickle.load(f)

winner       = weighted["winner"]        # e.g. "L1+Tanimoto"
best_alpha   = weighted["best_alpha"]
best_k_dist  = weighted["winner_k"]
best_sil_dist = weighted["winner_sil"]

logger.info("Weighted winner : %s  alpha=%.4f  k=%d  sil=%.4f",
            winner, best_alpha, best_k_dist, best_sil_dist)
print(f"\nLoaded from weighted notebook:")
print(f"  Winner config : {winner}")
print(f"  Best alpha    : {best_alpha:.4f}")
print(f"  Best k        : {best_k_dist}")
print(f"  Silhouette    : {best_sil_dist:.4f}")

# Parse winner string back to num_dist + cat_dist
# winner is stored as e.g. "L1+Tanimoto" or "Gower"
if winner == "Gower":
    use_gower    = True
    winner_num   = None
    winner_cat   = None
else:
    use_gower    = False
    winner_num, winner_cat = winner.split("+")

logger.info("use_gower=%s  num_dist=%s  cat_dist=%s",
            use_gower, winner_num, winner_cat)

2026-03-19 21:21:10,692 — INFO — Weighted winner : L1+Tanimoto  alpha=0.2000  k=10  sil=0.6924
2026-03-19 21:21:10,694 — INFO — use_gower=False  num_dist=L1  cat_dist=Tanimoto



Loaded from weighted notebook:
  Winner config : L1+Tanimoto
  Best alpha    : 0.2000
  Best k        : 10
  Silhouette    : 0.6924


This is the most important upstream finding. Alpha=0.2 means the categorical block (Job, Gender, Area) gets 80% of the weight, completely inverting the neutral baseline of 0.58. The sweep confirmed that on this dataset, categorical variables are dramatically more informative than their feature count implies. Job in particular (5 classes) is likely doing most of the work — a client's employment status is a much stronger signal of financial behaviour than marginal differences in continuous scores.

In [6]:
#  Compute the winning distance matrix
CACHE_DIR = Path("distance_cache")
CACHE_DIR.mkdir(exist_ok=True)

def load_or_compute(path: Path, compute_fn):
    if path.exists():
        logger.info("Loading cached matrix from %s", path)
        with open(path, "rb") as f:
            return pickle.load(f)
    logger.info("Computing matrix — will save to %s", path)
    D = compute_fn()
    with open(path, "wb") as f:
        pickle.dump(D, f)
    logger.info("Saved to %s", path)
    return D

if use_gower:
    cat_mask = build_gower_cat_mask(df, categorical_cols=categorical_cols)
    D_winner = load_or_compute(
        CACHE_DIR / "D_gower.pkl",
        lambda: gower_lib.gower_matrix(df.values, cat_features=cat_mask)
    )
else:
    X_cat = X_cat_ohe.copy() if winner_cat == "Tanimoto" else X_cat_int.copy()
    pkl_name = f"D_mixed_{winner_num.lower()}_{winner_cat.lower()}.pkl"
    # Try weighted cache first, fall back to recomputing
    weighted_cache = CACHE_DIR / "weighted" / pkl_name
    base_cache     = CACHE_DIR / pkl_name
    cache_path     = weighted_cache if weighted_cache.exists() else base_cache

    D_winner = load_or_compute(
        cache_path,
        lambda: mixed_distance_matrix(
            X_num, X_cat,
            alpha=best_alpha,
            num_dist=winner_num,
            cat_dist=winner_cat,
        )
    )

D_winner = D_winner.astype(np.float64)
logger.info("Winner distance matrix ready — shape=%s  min=%.4f  max=%.4f",
            D_winner.shape, D_winner.min(), D_winner.max())
print(f"\nDistance matrix shape : {D_winner.shape}")
print(f"min={D_winner.min():.4f}  max={D_winner.max():.4f}")

2026-03-19 21:21:10,717 — INFO — Loading cached matrix from distance_cache\weighted\D_mixed_l1_tanimoto.pkl
2026-03-19 21:21:10,736 — INFO — Winner distance matrix ready — shape=(495, 495)  min=0.0000  max=0.7864



Distance matrix shape : (495, 495)
min=0.0000  max=0.7864


In [7]:
#UMAP embedding configurations
#
# We test three dimensionalities for the clustering embedding.
# All use:
#   min_dist = 0.0  → tight packing, maximises cluster separability
#   n_neighbors = 30 → captures more global structure (appropriate for 5k)
#
# These are intentionally different from the 2D visualisation UMAP
# which uses min_dist=0.1 and n_components=2 to spread points for readability.

UMAP_CLUSTER_CONFIGS = {
    "umap_8d":  (8,  30, 0.0),
    "umap_12d": (12, 30, 0.0),
    "umap_15d": (15, 30, 0.0),
}

K_RANGE_UMAP      = range(2, 11)
N_STABILITY_SEEDS = 5
CACHE_DIR_UMAP    = Path("umap_cache")
CACHE_DIR_UMAP.mkdir(exist_ok=True)

logger.info("UMAP configs     : %s", list(UMAP_CLUSTER_CONFIGS.keys()))
logger.info("k range          : %s", list(K_RANGE_UMAP))
logger.info("Stability seeds  : %d", N_STABILITY_SEEDS)

2026-03-19 21:21:10,749 — INFO — UMAP configs     : ['umap_8d', 'umap_12d', 'umap_15d']
2026-03-19 21:21:10,751 — INFO — k range          : [2, 3, 4, 5, 6, 7, 8, 9, 10]
2026-03-19 21:21:10,752 — INFO — Stability seeds  : 5


In [8]:
# Build UMAP embeddings (cached)
umap_embeddings = {}

for config_name, (n_comp, n_neigh, m_dist) in UMAP_CLUSTER_CONFIGS.items():
    safe_winner = winner.replace(" ", "_").replace("/", "_").replace("+", "_")
    cache_path  = CACHE_DIR_UMAP / f"emb_{safe_winner}_{config_name}.pkl"

    if cache_path.exists():
        logger.info("Loading cached embedding: %s", config_name)
        with open(cache_path, "rb") as f:
            emb = pickle.load(f)
    else:
        logger.info("Computing UMAP embedding: %s  (%d-dim)", config_name, n_comp)
        reducer = umap.UMAP(
            n_components=n_comp,
            metric="precomputed",
            n_neighbors=n_neigh,
            min_dist=m_dist,
            random_state=42,
            low_memory=False,
        )
        emb = reducer.fit_transform(D_winner)
        with open(cache_path, "wb") as f:
            pickle.dump(emb, f)
        logger.info("Saved embedding to %s", cache_path)

    umap_embeddings[config_name] = emb
    print(f"  {config_name}: shape={emb.shape}")

print("\nAll UMAP embeddings ready.")

2026-03-19 21:21:10,762 — INFO — Computing UMAP embedding: umap_8d  (8-dim)
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
2026-03-19 21:21:20,265 — INFO — Saved embedding to umap_cache\emb_L1_Tanimoto_umap_8d.pkl
2026-03-19 21:21:20,268 — INFO — Computing UMAP embedding: umap_12d  (12-dim)
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
c:\Users\giuli\Repositories\fint

  umap_8d: shape=(495, 8)


2026-03-19 21:21:21,051 — INFO — Saved embedding to umap_cache\emb_L1_Tanimoto_umap_12d.pkl
2026-03-19 21:21:21,052 — INFO — Computing UMAP embedding: umap_15d  (15-dim)
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  umap_12d: shape=(495, 12)


2026-03-19 21:21:21,952 — INFO — Saved embedding to umap_cache\emb_L1_Tanimoto_umap_15d.pkl


  umap_15d: shape=(495, 15)

All UMAP embeddings ready.


In [9]:
# K-medoids clustering in UMAP latent space
#
# Clustering is done on Euclidean distance in the UMAP latent space,
# NOT on the original precomputed distance matrix. This is what makes
# UMAP-clustering a distinct method — it clusters the compressed
# representation. Silhouette is computed on the same Euclidean distance
# for consistency with what k-medoids optimised.

umap_results = {}

for config_name in UMAP_CLUSTER_CONFIGS:
    emb   = umap_embeddings[config_name]
    D_emb = cdist(emb, emb, metric="euclidean").astype(np.float64)

    metric_rows = []
    best_k, best_score, best_labels = None, -1, None

    for k in K_RANGE_UMAP:
        res  = kmedoids.fasterpam(D_emb, k, random_state=42)
        lbls = np.array(res.labels)

        sil = silhouette_score(D_emb, lbls, metric="precomputed")
        db  = davies_bouldin_score(D_emb, lbls)
        ch  = calinski_harabasz_score(D_emb, lbls)

        metric_rows.append({
            "k":                 k,
            "Silhouette":        round(sil, 4),
            "Davies-Bouldin":    round(db,  4),
            "Calinski-Harabasz": round(ch,  2),
            "labels":            lbls,
        })

        if sil > best_score:
            best_score, best_k, best_labels = sil, k, lbls

    umap_results[config_name] = {
        "best_k":      best_k,
        "best_score":  best_score,
        "best_labels": best_labels,
        "metrics":     pd.DataFrame(metric_rows).drop(columns=["labels"]),
        "all_labels":  {r["k"]: r["labels"] for r in metric_rows},
    }
    logger.info("%s → best k=%d  silhouette=%.4f", config_name, best_k, best_score)
    print(f"  {config_name}  →  best k={best_k}  silhouette={best_score:.4f}")

2026-03-19 21:21:22,158 — INFO — umap_8d → best k=8  silhouette=0.8413
2026-03-19 21:21:22,252 — INFO — umap_12d → best k=6  silhouette=0.8476
2026-03-19 21:21:22,341 — INFO — umap_15d → best k=5  silhouette=0.8157


  umap_8d  →  best k=8  silhouette=0.8413
  umap_12d  →  best k=6  silhouette=0.8476
  umap_15d  →  best k=5  silhouette=0.8157


Silhouette above 0.8 indicates very dense, well-separated clusters. For context, the distance-space best was 0.69 (also already very good). The jump to 0.84-0.85 in UMAP space reflects that the latent embedding has compressed the data in a way that makes the cluster boundaries even sharper. However — and this is critical — remember from the markdown: these scores are not comparable to the distance-space scores. They are computed in different spaces.

In [10]:
# Stability check across random seeds
#
# UMAP is stochastic — different seeds produce different embeddings.
# ARI >= 0.85 between seeds means the cluster structure is stable and
# does not depend on the random initialisation. Below 0.85 the result
# should be treated with caution.

stability_scores = {}

for config_name, (n_comp, n_neigh, m_dist) in UMAP_CLUSTER_CONFIGS.items():
    best_k   = umap_results[config_name]["best_k"]
    ari_list = []

    seed_labels = []
    for seed in range(N_STABILITY_SEEDS):
        reducer = umap.UMAP(
            n_components=n_comp, metric="precomputed",
            n_neighbors=n_neigh, min_dist=m_dist,
            random_state=seed, low_memory=False,
        )
        emb_s = reducer.fit_transform(D_winner)
        D_s   = cdist(emb_s, emb_s, metric="euclidean").astype(np.float64)
        res_s = kmedoids.fasterpam(D_s, best_k, random_state=seed)
        seed_labels.append(np.array(res_s.labels))

    for i in range(len(seed_labels)):
        for j in range(i + 1, len(seed_labels)):
            ari_list.append(adjusted_rand_score(seed_labels[i], seed_labels[j]))

    mean_ari = float(np.mean(ari_list))
    stability_scores[config_name] = mean_ari
    flag = "STABLE" if mean_ari >= 0.85 else "UNSTABLE"
    logger.info("%s  k=%d  ARI=%.3f  %s", config_name, best_k, mean_ari, flag)
    print(f"  {config_name}  k={best_k}  ARI={mean_ari:.3f}  {flag}")

c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv

  umap_8d  k=8  ARI=0.936  STABLE


c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv

  umap_12d  k=6  ARI=0.981  STABLE


c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv

  umap_15d  k=5  ARI=0.923  STABLE


ARI of 0.981 across 5 seeds means the 12-dimensional embedding produces virtually identical cluster assignments regardless of random initialisation. This is exceptional stability and means you can report this result with confidence — it is not a lucky seed.

In [11]:
# Validation metric plots
for config_name in UMAP_CLUSTER_CONFIGS:
    res  = umap_results[config_name]
    mdf  = res["metrics"]
    stab = stability_scores[config_name]

    fig = make_subplots(rows=1, cols=3, subplot_titles=[
        "Silhouette (↑ better)",
        "Davies-Bouldin (↓ better)",
        "Calinski-Harabasz (↑ better)",
    ])
    for ci, (col, color) in enumerate([
        ("Silhouette",        "#1f77b4"),
        ("Davies-Bouldin",    "#ff7f0e"),
        ("Calinski-Harabasz", "#2ca02c"),
    ]):
        fig.add_trace(go.Scatter(
            x=mdf["k"].tolist(), y=mdf[col].tolist(),
            mode="lines+markers", line=dict(color=color, width=2),
            marker=dict(size=8), showlegend=False,
        ), row=1, col=ci + 1)
        fig.add_vline(x=res["best_k"], line_dash="dash",
                      line_color="red", row=1, col=ci + 1)

    fig.update_xaxes(title_text="k")
    fig.update_layout(
        height=380,
        title_text=(
            f"UMAP clustering — {winner} / {config_name} "
            f"(best k={res['best_k']}  stability ARI={stab:.2f})"
        ),
    )
    fig.show()
    display(mdf)

,k,Silhouette,Davies-Bouldin,Calinski-Harabasz
0,2,0.5467,0.5099,558.01
1,3,0.7077,0.4366,1432.50
2,4,0.8085,0.2918,6274.07
3,5,0.7868,0.4047,6990.37
4,6,0.7617,0.4826,8091.89
5,7,0.8083,0.3318,8791.50
6,8,0.8413,0.2510,13646.18
7,9,0.7814,0.2922,15230.25
8,10,0.6905,0.4026,18454.76


,k,Silhouette,Davies-Bouldin,Calinski-Harabasz
0,2,0.5561,0.7905,766.37
1,3,0.6866,0.4936,1129.78
2,4,0.7944,0.2978,4838.03
3,5,0.7964,0.3818,6224.26
4,6,0.8476,0.2074,10175.52
5,7,0.8021,0.3305,12536.99
6,8,0.7338,0.3658,12951.55
7,9,0.6357,0.5163,13280.30
8,10,0.6734,0.4630,16858.65


,k,Silhouette,Davies-Bouldin,Calinski-Harabasz
0,2,0.4768,0.7239,402.51
1,3,0.6625,0.4820,1374.79
2,4,0.7942,0.2951,7259.97
3,5,0.8157,0.3108,8838.75
4,6,0.7757,0.4178,9986.59
5,7,0.7997,0.3351,12132.78
6,8,0.7351,0.3651,12958.94
7,9,0.7608,0.3469,15306.60
8,10,0.7867,0.2837,17252.01


Highest silhouette (0.848) combined with highest stability (0.981) and a sensible k=6. The 15-dimensional embedding produces fewer clusters (k=5) with lower silhouette — adding more dimensions does not help, suggesting the meaningful structure is captured well by 12 dimensions. The 8-dimensional embedding finds k=8 which may be over-splitting some natural groups.

In [12]:
# 2D visualisation of each UMAP configuration
#
# Clustering was done in high-dimensional UMAP space (8–15 dims).
# For visualisation we project the high-dim embedding down to 2D
# with a second UMAP call using min_dist=0.1 to spread points.
# We project the embedding, NOT the original distance matrix, so
# the 2D layout preserves the geometry found at high dimension.

for config_name in UMAP_CLUSTER_CONFIGS:
    res  = umap_results[config_name]
    lbl  = res["best_labels"]
    k    = res["best_k"]
    emb  = umap_embeddings[config_name]
    stab = stability_scores[config_name]

    viz_reducer = umap.UMAP(
        n_components=2, n_neighbors=15,
        min_dist=0.1, random_state=42,
    )
    emb_2d = viz_reducer.fit_transform(emb)

    df_viz = pd.DataFrame({
        "UMAP1":   emb_2d[:, 0],
        "UMAP2":   emb_2d[:, 1],
        "Cluster": [f"Cluster {c}" for c in lbl.tolist()],
    })

    fig = px.scatter(
        df_viz, x="UMAP1", y="UMAP2", color="Cluster",
        title=f"UMAP 2D — {winner} / {config_name}  (k={k}  ARI={stab:.2f})",
        opacity=0.6,
        color_discrete_sequence=px.colors.qualitative.Vivid,
    )
    fig.update_traces(marker=dict(size=4))
    fig.update_layout(height=520)
    fig.show()

c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [13]:
# elect best UMAP configuration
#
# Selection rule: highest silhouette among stable configs (ARI >= 0.85).
# If nothing is stable, fall back to highest silhouette and log a warning
# so the instability is visible in the output.

candidates = []
for config_name in UMAP_CLUSTER_CONFIGS:
    res  = umap_results[config_name]
    stab = stability_scores[config_name]
    candidates.append({
        "config":     config_name,
        "k":          res["best_k"],
        "silhouette": res["best_score"],
        "stability":  round(stab, 4),
        "stable":     stab >= 0.85,
    })

cand_df = pd.DataFrame(candidates).sort_values(
    ["stable", "silhouette"], ascending=[False, False]
).reset_index(drop=True)

print("\n── UMAP configuration ranking ──")
display(cand_df)

best_row        = cand_df.iloc[0]
best_config     = best_row["config"]
best_k_umap     = int(best_row["k"])
best_sil_umap   = float(best_row["silhouette"])
best_stab_umap  = float(best_row["stability"])

if not best_row["stable"]:
    logger.warning(
        "No stable UMAP config found (ARI < 0.85). "
        "Selecting best silhouette regardless: %s  ARI=%.3f",
        best_config, best_stab_umap,
    )

logger.info("Selected: %s  k=%d  sil=%.4f  stability=%.3f",
            best_config, best_k_umap, best_sil_umap, best_stab_umap)

print(f"\nSelected : {winner} / {best_config}")
print(f"  k          : {best_k_umap}")
print(f"  Silhouette : {best_sil_umap:.4f}")
print(f"  Stability  : {best_stab_umap:.3f}")

best_labels_umap = umap_results[best_config]["best_labels"]
df["cluster_umap"] = best_labels_umap

print(f"\nCluster size distribution:")
print(df["cluster_umap"].value_counts().sort_index().to_string())


── UMAP configuration ranking ──


,config,k,silhouette,stability,stable
0,umap_12d,6,0.847599,0.9814,True
1,umap_8d,8,0.841263,0.9364,True
2,umap_15d,5,0.815710,0.9227,True


2026-03-19 21:21:41,910 — INFO — Selected: umap_12d  k=6  sil=0.8476  stability=0.981



Selected : L1+Tanimoto / umap_12d
  k          : 6
  Silhouette : 0.8476
  Stability  : 0.981

Cluster size distribution:
cluster_umap
0     17
1    115
2     48
3    128
4     56
5    131


In [14]:
# Profile UMAP clusters
JOB_MAP  = {1: "Unemployed", 2: "Employee", 3: "Manager",
             4: "Entrepreneur", 5: "Retired"}
GEN_MAP  = {0: "Male", 1: "Female"}
AREA_MAP = {1: "North", 2: "Center", 3: "South"}

KEY_FEATURES     = ["Income", "Wealth", "Debt", "Saving", "Luxury", "FinEdu"]
PAL              = px.colors.qualitative.Vivid
umap_method_name = f"UMAP ({winner} / {best_config})"

df_p             = df.copy()
df_p["Cluster"]  = df_p["cluster_umap"]
num_profile_cols = [c for c in numerical_cols if c in df_p.columns]

# ── Summary table ─────────────────────────────────────────────────────────────
rows = []
for c in range(best_k_umap):
    g   = df_p[df_p["Cluster"] == c]
    row = {"Cluster": c, "N": len(g), "%": f"{len(g)/len(df_p)*100:.1f}%"}
    for col in num_profile_cols:
        row[col] = round(float(g[col].mean()), 3)
    row["Job"]    = JOB_MAP.get(int(g["Job"].mode()[0]),    "?")
    row["Gender"] = GEN_MAP.get(int(g["Gender"].mode()[0]), "?")
    row["Area"]   = AREA_MAP.get(int(g["Area"].mode()[0]),  "?")
    rows.append(row)

print(f"\n{'='*60}")
print(f" {umap_method_name} — k={best_k_umap} cluster summary")
print(f"{'='*60}")
display(pd.DataFrame(rows))

# ── Radar chart ───────────────────────────────────────────────────────────────
radar = go.Figure()
for c in range(best_k_umap):
    g    = df_p[df_p["Cluster"] == c]
    vals = [float(g[col].mean()) for col in num_profile_cols]
    radar.add_trace(go.Scatterpolar(
        r=vals + [vals[0]],
        theta=num_profile_cols + [num_profile_cols[0]],
        fill="toself", name=f"Cluster {c}",
        line_color=PAL[c % len(PAL)], opacity=0.75,
    ))
radar.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title=f"Radar — {umap_method_name} (k={best_k_umap})",
    height=500,
)
radar.show()

# ── Boxplots for key features ─────────────────────────────────────────────────
box = make_subplots(rows=1, cols=len(KEY_FEATURES), subplot_titles=KEY_FEATURES)
for ci, col in enumerate(KEY_FEATURES):
    for c in range(best_k_umap):
        g = df_p[df_p["Cluster"] == c]
        box.add_trace(go.Box(
            y=g[col].tolist(), name=f"Cluster {c}",
            marker_color=PAL[c % len(PAL)], showlegend=(ci == 0),
        ), row=1, col=ci + 1)
box.update_layout(height=420, boxmode="group",
                  title_text=f"Key Features by Cluster — {umap_method_name}")
box.show()


 UMAP (L1+Tanimoto / umap_12d) — k=6 cluster summary


,Cluster,N,%,Age,CitySize,FamilySize,Income,Wealth,Debt,FinEdu,ESG,Digital,BankFriend,LifeStyle,Luxury,Saving,Investments,Job,Gender,Area
0,0,17,3.4%,58.412,1.647,3.118,0.500,0.564,0.515,0.517,0.527,0.524,0.557,0.422,0.414,0.520,2.294,Employee,Male,South
1,1,115,23.2%,69.235,1.800,2.304,0.509,0.513,0.329,0.457,0.593,0.424,0.623,0.352,0.408,0.453,2.061,Retired,Female,North
2,2,48,9.7%,74.083,1.625,2.333,0.455,0.432,0.242,0.417,0.609,0.371,0.688,0.357,0.337,0.376,1.854,Retired,Male,North
3,3,128,25.9%,56.836,2.039,2.391,0.588,0.614,0.489,0.542,0.584,0.572,0.605,0.512,0.511,0.577,2.211,Employee,Female,North
4,4,56,11.3%,56.857,1.964,2.589,0.595,0.635,0.429,0.531,0.560,0.548,0.656,0.507,0.511,0.491,2.161,Employee,Male,Center
5,5,131,26.5%,57.496,2.008,2.687,0.627,0.628,0.507,0.568,0.576,0.588,0.637,0.516,0.570,0.597,2.397,Employee,Male,North


In [15]:
# Save results for comparison notebook
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

# Project best high-dim embedding to 2D for the comparison notebook's
# side-by-side visualisation (Cell 6 of the comparison notebook)
best_emb_highdim = umap_embeddings[best_config]
viz_reducer      = umap.UMAP(
    n_components=2, n_neighbors=15, min_dist=0.1, random_state=42,
)
emb2d_final = viz_reducer.fit_transform(best_emb_highdim)

pickle.dump({
    "umap_lbls":        best_labels_umap,
    "umap_k":           best_k_umap,
    "umap_sil":         best_sil_umap,
    "umap_stability":   best_stab_umap,
    "config_used":      best_config,
    "dist_used":        winner,
    "alpha_used":       best_alpha,
    "emb2d":            emb2d_final,
    "umap_results":     umap_results,
    "stability_scores": stability_scores,
}, open(RESULTS_DIR / "umap_results.pkl", "wb"))

logger.info("Saved umap_results.pkl — k=%d  sil=%.4f  stability=%.3f",
            best_k_umap, best_sil_umap, best_stab_umap)
print("Saved umap_results.pkl")

c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
2026-03-19 21:21:42,869 — INFO — Saved umap_results.pkl — k=6  sil=0.8476  stability=0.981


Saved umap_results.pkl


Clusters 1, 3, and 5 account for 75.6% of clients and are roughly equal in size — these are your three dominant segments. Clusters 2 and 4 are medium-sized minority segments. Cluster 0 with only 17 clients (3.4%) is worth investigating carefully. It could be a genuinely distinctive niche segment, or it could be an artefact — clients that do not fit cleanly into any of the five main groups and get isolated by UMAP's local neighbourhood structure. Before reporting it as a segment, check whether those 17 clients share a coherent profile or whether they are simply outliers that survived the Isolation Forest step.

The critical number will be the ARI between the weighted distance-space labels (k=10, sil=0.69) and the UMAP labels (k=6, sil=0.85). These have different k values — 10 vs 6 — which means ARI will structurally be lower than if they agreed on k, because some of the weighted notebook's 10 clusters may map onto single UMAP clusters. This is not necessarily a problem: it may mean the UMAP compression merged some genuinely similar sub-groups that the distance-space method split finely. Look at which weighted clusters merge together in the UMAP solution — that pattern is itself a finding about which sub-groups are most similar to each other.